In [0]:
%run ./gold_functions

In [0]:
print(build_dim_date(), "rows in dim_date")

In [0]:
%sql
SELECT full_date, day_name, is_trading_day, policy_rate_pct, rate_change_bps, is_policy_decision_day
FROM workspace.gold.dim_date
WHERE full_date BETWEEN '2023-06-24' AND '2023-06-30'
ORDER BY full_date

In [0]:
%sql
Select Count(*)
from workspace.gold.dim_date 

In [0]:
print(build_dim_stock(), "rows in dim_stock")

In [0]:
%sql
SELECT security_type, sector_name, COUNT(*) AS stocks, SUM(CASE WHEN is_active THEN 1 ELSE 0 END) AS active
FROM workspace.gold.dim_stock
GROUP BY security_type, sector_name
ORDER BY security_type, stocks DESC

In [0]:
print(build_fact_daily_stock("2015-01-01", "2026-09-29"))

In [0]:
%sql
SELECT d.year, COUNT(*) AS rows,
       ROUND(AVG(f.vol_20d), 3) AS avg_vol_20d,
       ROUND(AVG(CASE WHEN f.risk_tier = 'High' THEN 1.0 ELSE 0 END), 3) AS share_high,
       SUM(CASE WHEN f.is_extreme_move THEN 1 ELSE 0 END) AS extreme_moves
FROM workspace.gold.fact_daily_stock f
JOIN workspace.gold.dim_date d ON f.date_key = d.date_key
GROUP BY d.year ORDER BY d.year

In [0]:
print(build_fact_market_daily("2015-01-01", "2026-09-29"))

In [0]:
%sql
SELECT d.year,
       ROUND(AVG(m.market_vol_20d), 3) AS avg_market_vol,
       ROUND(MAX(m.market_vol_20d), 3) AS peak_market_vol,
       ROUND(AVG(m.median_stock_vol_20d), 3) AS typical_stock_vol,
       ROUND(AVG(d.policy_rate_pct), 2) AS avg_policy_rate
FROM workspace.gold.fact_market_daily m
JOIN workspace.gold.dim_date d ON m.date_key = d.date_key
GROUP BY d.year ORDER BY d.year